In [ ]:
### conda activate [] environment with scvi-tools, scanpy, matplotlib, numpy, torch installed

import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import numpy as np

import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


Combine all adatas; h5ad objects not provided but can be recreated from cellranger outputs (GEO) using:
- 00_example_cellranger_to_filtered_clustered.ipynb - example to go from cellranger output to processed (filtered, clustered, etc) h5ad 
- ../../3_figure_FFPE/code/00_probabilistic_genotype_assignment - example to go from processed h5ad to genotype assignments

In [ ]:
dir = '' ### set to directory where data is stored

adatas = []
samples = []
for lib in ['1-plex','4-plex','1','2']:
    if lib == '4-plex':
        BCs = ['BC001', 'BC002', 'BC003', 'BC004']
    elif lib == '1-plex':
        BCs = ['BC001']
    else:
        BCs = ['BC001', 'BC002', 'BC003', 'BC004', 'BC005', 'BC006', 'BC007', 'BC008', 'BC009', 'BC010', 'BC011', 'BC012', 'BC013', 'BC014', 'BC015', 'BC016']
    for BC in BCs:
        if lib == '4-plex':
            adata_dir = dir + 'MPN_4plex_' + BC + '_genotyped.h5ad'
        elif lib == '1-plex':
            adata_dir = dir + 'MPN_1plex_' + BC + '_genotyped.h5ad'
        else:
            adata_dir = dir + 'MPN_' + lib + '_' + BC + '_genotyped.h5ad'
        temp_adata = sc.read_h5ad(adata_dir)

        samples.append(BC + '_' + lib)
        print(BC + '_' + lib, temp_adata.shape)
        adatas.append(temp_adata)

adata = sc.concat(
    adatas,
    label='sample',
    keys=samples,
    index_unique='-',
    join='outer'
)
del adatas

# Find HVGs and run SCVI

In [ ]:
adata.X = adata.layers['raw_data'].copy()  # use raw data for analysis
sc.pp.normalize_total(adata)
sc.pp.log1p(adata, base=2)

adata.obs['patient'] = adata.obs['sample'].copy()

sc.pp.highly_variable_genes(
    adata, n_top_genes=3000, inplace=True, subset=False, flavor="seurat_v3", batch_key='sample', layer='raw_data'
)
adata = adata[:,(adata.var['highly_variable_nbatches'] >= 10)].copy()


In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="raw_data",
    categorical_covariate_keys=["sample"],
    continuous_covariate_keys=["mito_frac"],
)


In [ ]:
model = scvi.model.SCVI(adata, gene_likelihood='nb')
model.train(batch_size=256, early_stopping=True, max_epochs=200)

In [ ]:
latent = model.get_latent_representation()
adata.obsm["X_scVI"] = latent
adata.layers["scvi_normalized"] = model.get_normalized_expression(library_size=1e4)


In [ ]:
# use scVI latent space for UMAP generation
sc.pp.neighbors(adata, use_rep="X_scVI")
sc.tl.umap(adata, min_dist=0.3)



In [ ]:
### write adata and model to file
### provided as Integrated_MPN_scvi_genotyped_nb.h5ad